In [ ]:
import serial
import time
import random

# Function to send AT command

In [ ]:
def send_command(command):
    ser.write((command + '\r\n').encode())
    time.sleep(1)
    response = ser.read(ser.inWaiting()).decode()
    #print("Response:", response)  # Print response for debugging
    return response

# Function to connect to MQTT broker

In [ ]:
def connect_to_mqtt(host, port, user, password):
    send_command('AT+CGATT=1')
    send_command('AT+CGDCONT=1,"IP","CMNET"')
    send_command('AT+CGACT=1,1')
    command = 'AT+MQTTCONN="{}",{},12345,120,0,"{}","{}"'.format(host, port, user, password)
    send_command(command)

# Function to publish message to MQTT broker

In [ ]:
def publish_message(topic, message):
    command = 'AT+MQTTPUB="{}","{}",0,0,0'.format(topic, message)
    send_command(command)

# Function to disconnect from MQTT broker

In [ ]:
def disconnect_mqtt():
    send_command('AT+MQTTDISCONN')

# Main code

In [ ]:
if __name__ == "__main__":
    # Open serial connection to the A9G module
    ser = serial.Serial('COM5', 115200, timeout=1)

    # Wait for the module to initialize
    time.sleep(2)

    # MQTT broker details
    mqtt_host = "0.tcp.in.ngrok.io"
    mqtt_port = "12130"
    mqtt_user = ""
    mqtt_password = ""
    mqtt_topic = "Test"  # Topic to publish messages


    # Connect to MQTT broker
    print("Connecting to MQTT broker...")
    connect_to_mqtt(mqtt_host, mqtt_port, mqtt_user, mqtt_password)
    print("Connected to MQTT broker")

    try:
        while True:
            # Generate random value to publish
            random_value = random.randint(0, 100)
            # Publish random value to MQTT broker
            print("Publishing value:", random_value)
            publish_message(mqtt_topic, str(random_value))
            print("Published value:", random_value)
            time.sleep(5)  # Publish every 5 seconds

    except KeyboardInterrupt:
        print("Disconnecting from MQTT broker...")
        # Disconnect from MQTT broker
        disconnect_mqtt()
        print("Disconnected from MQTT broker")

    # Close serial connection
    ser.close()